# BioiOSMutant — Biomutant (Switch → iOS 26)

**Title ID:** `01004BA017CD6000` · **branch:** `BioiOSMutant` · **Drive:** `biotant`.

**Ordem:** 1) autenticar Drive; 2) clonar branch; 3) baixar e extrair os dois RARs, enviar **apenas NSPs** ao Drive, apagar RARs; 4) recuperar checkpoint se existir; 5) preparar o jogo em Linux; 6) conferir; 7) enviar handoff ao GitHub; 8) após sucesso do Actions, transferir IPA ao Drive.

O Colab realiza todos os trabalhos pesados que podem ser feitos em Linux. O Actions usa **xcode-27** apenas para host Apple Silicon, Metal, framework e assinatura ad-hoc. A IPA contém os shaders compilados e usa `romfs.img` externa. **Nenhum arquivo NSP ou chave entra no GitHub nem na IPA.**


## 1. Autenticar o Drive
Pasta principal já configurada; o jogo terá uma subpasta exclusiva `biotant`.

In [ ]:
!python -m pip -q install google-api-python-client google-auth-httplib2 requests
from google.colab import auth
import google.auth
from googleapiclient.discovery import build
from pathlib import Path
import os, sys, json, re, argparse, hashlib, subprocess, importlib, getpass

auth.authenticate_user()
credentials, _ = google.auth.default(scopes=["https://www.googleapis.com/auth/drive"])
DRIVE_CLIENT = build("drive", "v3", credentials=credentials, cache_discovery=False)
DRIVE_PARENT_ID = "19HXEJCK-26NfBVKeUP3wqFxMFpoF3ykm"
BIOTANT_FOLDER_ID = "1yrL87jYVf08FGB2eWV5CY0jBp2OTHYAs"
parent = DRIVE_CLIENT.files().get(fileId=DRIVE_PARENT_ID, fields="id,name,mimeType", supportsAllDrives=True).execute()
child = DRIVE_CLIENT.files().get(fileId=BIOTANT_FOLDER_ID, fields="id,name,mimeType,parents", supportsAllDrives=True).execute()
assert child["name"] == "biotant" and DRIVE_PARENT_ID in child["parents"], "Pasta biotant incorreta"
print("Drive autenticado:", parent["name"], "/ biotant")


## 2. Obter scripts atualizados da branch BioiOSMutant
Não usar os scripts `blades/`. O clone não baixa NSPs.

In [ ]:
REPO = Path("/content/BioiOSMutant-repo")
if not (REPO / ".git").is_dir():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "BioiOSMutant",
                    "https://github.com/joao9vegeto-stack/KintTany.git", str(REPO)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO), "fetch", "--depth", "1", "origin", "BioiOSMutant"], check=True)
    subprocess.run(["git", "-C", str(REPO), "reset", "--hard", "FETCH_HEAD"], check=True)
COLAB_SRC = str(REPO / "biotant" / "colab")
sys.path.insert(0, COLAB_SRC)
COMMIT = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
print("Branch BioiOSMutant / commit:", COMMIT)


## 3. Importar os RARs base + update para o Drive (Colab)

**Download em chunks:** 6 workers HTTP Range, 64 MiB por bloco, verificação SHA-256 por bloco e retomada de blocos já verificados. Se o link não aceitar Range, usa streaming seguro de 8 MiB. O fluxo extrai um RAR por vez, verifica NSP/PFS0, envia ao Drive em chunks resumíveis de 64 MiB (tamanho/MD5 confirmado) e apaga o RAR somente depois do upload validado. O padrão de seis workers também é usado em Drive → Colab e Drive → Actions. URLs temporárias podem expirar.


In [ ]:
# Testes offline do downloader antes de transferir gigabytes.
subprocess.run(["python3", "-m", "unittest", "discover",
                "-s", str(REPO / "biotant" / "tests"), "-p", "test_*.py", "-v"], check=True)
import archive_ingest
BASE_RAR_URL = "https://e2.megaupdownup.org/0b7d9dd6477c031d562610d2c7e5eca6/Biomutant_(NSP)(eShop).rar?download_token=dbfa03987e1d651d5bf9b4128f1d9eb64fab181a4551507147eb0134301dcd76"
UPDATE_RAR_URL = "https://e17.megaupdownup.org/929e4e3ca5910dd4a7699627d7bd2f61/BOIMUTNT-(USA)-NSwTcH-NSP-Update102-Ziperto.rar?download_token=c7b39cb90010e191130f99ae6de4f46e86d73cc46e843b9845236497cb970933"
SOURCE_IMPORT = archive_ingest.ingest(
    DRIVE_CLIENT, DRIVE_PARENT_ID, BASE_RAR_URL, UPDATE_RAR_URL,
    root="/content/biomutant")
print(json.dumps(SOURCE_IMPORT, indent=2, ensure_ascii=False))


## 4. Recuperar checkpoint Linux do Drive, se disponível
Evita baixar NSPs e recompilar trabalho já finalizado quando existe handoff válido de Biomutant.

In [ ]:
import run_colab
RECOVERED_HANDOFF = run_colab.recover(argparse.Namespace(drive_folder_id=DRIVE_PARENT_ID), api=DRIVE_CLIENT)
print("Checkpoint recuperado:", bool(RECOVERED_HANDOFF))


## 5. Construir perfil real a partir dos NSPs e preparar Linux
`build_ipa_from_nsp.py` verifica Title ID e versão de update, levanta perfil provisório quando não existir, faz merge RomFS, rewrite ARM64, split e decode de shaders quando suportado em Linux, cria workspace verificável, e envia RomFS externa + checkpoint ao Drive. Só operações Apple Silicon/Metal/Xcode ficam para Actions.

In [ ]:
subprocess.run(["git", "-C", str(REPO), "fetch", "--depth", "1", "origin", "BioiOSMutant"], check=True)
subprocess.run(["git", "-C", str(REPO), "reset", "--hard", "FETCH_HEAD"], check=True)
importlib.invalidate_caches()
run_colab = importlib.reload(run_colab)
assert run_colab.BRANCH == "BioiOSMutant" and run_colab.TITLE_ID == "01004BA017CD6000"
if RECOVERED_HANDOFF:
    HANDOFF = RECOVERED_HANDOFF
    print("Reutilizando checkpoint Drive; nenhuma etapa Linux refeita.")
else:
    HANDOFF = run_colab.prepare(argparse.Namespace(drive_folder_id=DRIVE_PARENT_ID), api=DRIVE_CLIENT)
HANDOFF_PATH = Path("/content/biomutant-handoff.json")
assert HANDOFF_PATH.is_file(), "Handoff ainda não foi criado"
print("Preparação concluída:", HANDOFF["request_id"])


## 6. Gate de identidade e integridade antes do Actions
Não inventar versão do update, backend gráfico, quantidade de shaders ou perfil. Os dados devem vir do NSP real.

In [ ]:
HANDOFF = json.loads(HANDOFF_PATH.read_text())
p = HANDOFF["prepared"]
assert HANDOFF["project"] == HANDOFF["branch"] == "BioiOSMutant"
assert HANDOFF["title_id"] == p["title_id"] == "01004BA017CD6000"
assert p["input_package_count"] == p["input_nsp_count"] == 2
assert p["input_update_count"] == 1 and p["input_dlc_count"] == 0
assert p["base_content_version"] == 0
assert p["update_content_version"] > 0 and HANDOFF["package_version"] == f'v{p["update_content_version"]}'
assert p["graphics"] in ("nvn","gl","vulkan")
assert HANDOFF["game_data"]["romfs"]["size"] > 0
for item in (HANDOFF["source_files"]["base"], HANDOFF["source_files"]["update"], HANDOFF["work_archive"], HANDOFF["game_data"]["romfs"]):
    assert re.fullmatch("[0-9a-f]{64}", item["sha256"])
print("GATE OK; backend:", p["graphics"], "update:", HANDOFF["package_version"])
print("Perfil provisório:", p["provisional"], "shader:", p["shader_split_stage"])
print("Mac stages reais:", p["mac_host_stages"])
print("RomFS externa:", HANDOFF["game_data"]["folder_url"])


## 7. Disparar **um** workflow macOS
Publica **somente** `biotant/handoff.json` em `BioiOSMutant`. O push aciona apenas `biomutant.yml`. Use PAT fine-grained com Contents: Read and write para KintTany; o token não é gravado no notebook.

In [ ]:
if not os.environ.get("BIOMUTANT_GITHUB_TOKEN"):
    os.environ["BIOMUTANT_GITHUB_TOKEN"] = getpass.getpass("GitHub PAT (Contents: R/W): ").strip()
assert os.environ["BIOMUTANT_GITHUB_TOKEN"]
run_colab.dispatch(argparse.Namespace(handoff=HANDOFF_PATH))
print("Actions: https://github.com/joao9vegeto-stack/KintTany/actions/workflows/biomutant.yml")


## 8. GitHub → Colab → Google Drive/biotant/ipa
Execute **somente depois** de um Actions bem-sucedido. Localiza a IPA da branch correta, confere o relatório e os hashes do artefato, e envia a IPA à pasta `biotant/ipa` com proveniência.

In [ ]:
import transfer_ipa
if not os.environ.get("BIOMUTANT_GITHUB_TOKEN"):
    os.environ["BIOMUTANT_GITHUB_TOKEN"] = getpass.getpass("GitHub PAT (Actions: Read): ").strip()
transfer_ipa.main()
